# Noir Mystery Story Engine: Project Overview

This notebook is the end-to-end walkthrough for the Noir Mystery Story Engine, an agentic AI system that turns a crime scene description into a structured noir mystery story. The system combines PydanticAI agents, typed Pydantic outputs, tool use, retrieval-augmented generation, persistent conversation memory, multi-agent orchestration, and a pydantic-evals evaluation suite.

The core idea is that a single prompt should not do all the work. The orchestrator delegates to specialist agents: a Detective identifies clues and suspects, a Witness retrieves supporting context from classic detective fiction or the web, and a Narrator turns the investigation into a polished story with a clear verdict.

## Requirements Covered

| Project requirement | Where it appears in this repository |
|---|---|
| PydanticAI agents | `src/agents/detective.py`, `src/agents/witness.py`, `src/agents/narrator.py` |
| Structured outputs | `src/models/clue_finding.py`, `src/models/story_report.py`, `OrchestratorOutput` |
| Tool use | `src/tools/clue_scorer.py`, `src/tools/web_search.py`, `src/tools/file_reader.py` |
| RAG | `src/rag/ingest.py`, `src/rag/retriever.py`, `chroma_store/`, `corpus/` |
| Conversation memory | `detective_memory.json`, managed by `src/agents/detective.py` |
| Multi-agent orchestration | `src/agents/orchestrator.py` |
| Evaluation suite | `src/evals/suite.py`, plus the evaluation summary image `evals.png` |
| Runnable application | `app.py` Shiny UI and `main.py` CLI |

## Architecture

```mermaid
flowchart TD
    A([User crime scene]) --> B[Orchestrator]
    B --> C[Detective Agent]
    B --> D[Witness Agent]
    B --> E[Narrator Agent]
    C --> F[(detective_memory.json)]
    D --> G[(ChromaDB noir_corpus)]
    G --> H[Classic detective corpus]
    D --> I[Web search fallback]
    C --> J[ClueFinding structured output]
    B --> K[clue_scorer tool]
    E --> L[StoryReport structured output]
    L --> M([Final noir story])
    M --> N[pydantic-evals]
```

The Detective creates a typed clue analysis and saves message history for later turns. The Witness queries the ChromaDB corpus first; if the retrieved passages are too weak, it falls back to web search. The Narrator uses the scene, clues, suspect, and witness context to produce the final `StoryReport`.

## Setup Notes

Run this notebook from the repository root. The expected setup is:

```bash
uv sync
cp .env.example .env
uv run python src/rag/ingest.py
```

The `.env` file should define the course gateway credentials:

```bash
OPENAI_API_KEY=your_course_gateway_key_here
OPENAI_BASE_URL=https://litellm.6640.ucf.spencerlyon.com
```

Cells that call LLM agents require working API credentials. The local inspection and deterministic tool cells can run without calling an LLM.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source exists: {(PROJECT_ROOT / 'src').exists()}")
print(f"Corpus files: {len(list((PROJECT_ROOT / 'corpus').glob('*.txt')))}")
print(f"Chroma store exists: {(PROJECT_ROOT / 'chroma_store').exists()}")
print(f"Evaluation image exists: {(PROJECT_ROOT / 'evals.png').exists()}")

## Example Input

The same input is used throughout this notebook so each component can be inspected independently before the full system runs.

In [ ]:
scene = """
A wealthy banker was found dead in his locked study at midnight.
The window was open despite the rain. A half-empty glass of whiskey
sat on the desk next to a torn letter. The butler claims he heard
nothing, but the cook saw a shadow near the garden at 11pm.
""".strip()

print(scene)

## Structured Output Models

The agents do not return loose text blobs. They return typed models that make downstream orchestration easier to validate and consume.

In [ ]:
from src.models.clue_finding import Clue, ClueFinding
from src.models.story_report import StoryReport

print("ClueFinding fields:", list(ClueFinding.model_fields))
print("StoryReport fields:", list(StoryReport.model_fields))
print("OrchestratorOutput fields: ['story', 'clues_found', 'top_clue', 'prime_suspect']")

## Tool Demonstration: Clue Scoring

`score_clues` is a deterministic tool used by the orchestrator after the detective identifies candidate clues. It ranks evidence so the final summary can report the most important clue.

In [ ]:
from src.tools.clue_scorer import score_clues

candidate_clues = [
    "open window despite heavy rain",
    "torn letter on the desk",
    "half-empty whiskey glass",
    "shadow seen in the garden",
]

for clue in score_clues(candidate_clues):
    print(f"{clue.score}/10 - {clue.description} ({clue.reason})")

## Tool Demonstration: File Reader

The file reader gives the system direct access to local case files in the `corpus/` directory. This is useful for deterministic inspection and for grounding the application in the same documents used by the RAG pipeline.

In [ ]:
from src.tools.file_reader import read_case_file

corpus_files = sorted(p.name for p in Path('corpus').glob('*.txt'))
print("Available corpus files:")
for name in corpus_files:
    print("-", name)

if corpus_files:
    sample = read_case_file(corpus_files[0])
    print("\nSample from", corpus_files[0])
    print(sample[:700])

## RAG Pipeline

The RAG pipeline chunks public-domain detective stories, embeds the chunks with `all-MiniLM-L6-v2`, and stores them in a persistent ChromaDB collection named `noir_corpus`. The Witness agent queries this collection before falling back to web search.

The chunking strategy is implemented in `src/rag/ingest.py`: 400-word chunks with 50-word overlap. This keeps passages long enough to preserve scene context while overlapping enough to reduce boundary loss.

In [ ]:
# This cell requires the vector store to be built with:
# uv run python src/rag/ingest.py

from src.rag.retriever import NightRetriever

try:
    retriever = NightRetriever(n_results=3)
    passages = retriever.query("locked room open window torn letter")
    for i, passage in enumerate(passages, start=1):
        print(f"Passage {i} from {passage['source']}:")
        print(passage['text'][:500].replace('\n', ' '))
        print()
except Exception as exc:
    print("RAG query could not run in this environment.")
    print(type(exc).__name__, exc)

## Agent 1: Detective

The Detective agent receives the crime scene and returns a `ClueFinding` model containing clues, locations, a prime suspect, reasoning, and confidence. It also persists message history in `detective_memory.json`, which gives the system conversation memory across turns.

In [ ]:
# Requires API credentials.
import asyncio
from src.agents.detective import investigate, MEMORY_FILE

async def demo_detective():
    finding = await investigate(scene)
    print(f"Prime suspect: {finding.prime_suspect}")
    print(f"Confidence: {finding.confidence}/10")
    print("Clues:")
    for clue in finding.clues:
        print(f"- {clue.description} at {clue.location} ({clue.suspicion_level}/10)")
    print("\nReasoning:")
    print(finding.reasoning)
    print(f"\nMemory file exists: {MEMORY_FILE.exists()}")

# await demo_detective()

## Agent 2: Witness

The Witness agent retrieves context relevant to the detective's clue summary. It checks the local corpus first; if retrieval is weak, it switches to web search. This gives the system a graceful fallback instead of forcing every case to rely on local documents.

In [ ]:
# Requires API credentials. Web fallback also requires network access.
async def demo_witness():
    from src.agents.witness import retrieve_context

    clue_summary = "locked room, open window, torn letter, whiskey glass, garden shadow"
    findings = await retrieve_context(clue_summary)
    print("Source type:", findings.source_type)
    print("Sources:")
    for source in findings.sources:
        print("-", source)
    print("\nRelevance summary:")
    print(findings.relevance_summary)

# await demo_witness()

## Agent 3: Narrator

The Narrator turns the structured detective findings and witness context into a `StoryReport`. The final model includes a title, setting, clue summary, prime suspect, verdict, and full story text.

In [ ]:
# Requires API credentials.
from pydantic import BaseModel

class NotebookWitnessFindings(BaseModel):
    passages: list[str]
    sources: list[str]
    relevance_summary: str
    source_type: str

async def demo_narrator_with_mock_inputs():
    from src.agents.narrator import narrate

    mock_clues = ClueFinding(
        clues=[
            Clue(description="the open rain-streaked window", suspicion_level=8, location="study"),
            Clue(description="the torn letter", suspicion_level=9, location="desk"),
            Clue(description="the half-empty whiskey glass", suspicion_level=6, location="desk"),
        ],
        prime_suspect="the butler",
        reasoning="The locked room points inward, while the torn letter suggests a personal motive.",
        confidence=8,
    )
    mock_witness = NotebookWitnessFindings(
        passages=["Classic detective stories often hide motive in letters and locked rooms."],
        sources=["local corpus"],
        relevance_summary="The retrieved passages connect locked rooms, letters, and trusted household suspects.",
        source_type="corpus",
    )
    story = await narrate(scene, mock_clues, mock_witness)
    print(story.model_dump_json(indent=2))

# await demo_narrator_with_mock_inputs()

## Full Orchestration

`run_mystery_engine` is the main application workflow. It runs the Detective, asks the Witness for context, scores the clues, runs the Narrator, and returns a compact `OrchestratorOutput` for the CLI, Shiny UI, and evaluation suite.

In [ ]:
# Requires API credentials.
async def demo_full_engine():
    from src.agents.orchestrator import run_mystery_engine

    result = await run_mystery_engine(scene)
    print(f"Title: {result.story.title}")
    print(f"Prime suspect: {result.prime_suspect}")
    print(f"Top clue: {result.top_clue}")
    print(f"Clues found: {result.clues_found}")
    print(f"Verdict: {result.story.verdict}")
    print("\nStory excerpt:")
    print(result.story.full_story[:1200])

# await demo_full_engine()

## Conversation Memory Check

The Detective persists message history to `detective_memory.json`. The exact size depends on how many agent calls have been made locally, but the presence of this file demonstrates that the agent has state outside a single request.

In [ ]:
memory_path = Path('detective_memory.json')
if memory_path.exists():
    raw_memory = json.loads(memory_path.read_text())
    print(f"Memory file: {memory_path}")
    print(f"Stored message records: {len(raw_memory)}")
    print("First record keys:", list(raw_memory[0]) if raw_memory else [])
else:
    print("No memory file yet. Run the Detective or full engine demo to create one.")

## Evaluation Suite

The evaluation suite contains 12 cases and three evaluator types:

- Deterministic: `ContainsExpectedFields` checks title, suspect, and story content.
- Behavioral: `CluesDiscovered` checks that at least three clues are found.
- LLM judge: `NoirQualityJudge` checks whether the story has noir atmosphere, a clear suspect, and a dramatic verdict.

The full suite calls the LLM multiple times, so it should be run when API credentials are configured and rate limits are available.

In [ ]:
try:
    from src.evals.suite import build_dataset

    dataset = build_dataset()
    print(f"Evaluation cases: {len(dataset.cases)}")
    print("Evaluators:", [type(evaluator).__name__ for evaluator in dataset.evaluators])
    print("Case names:")
    for case in dataset.cases:
        print("-", case.name)
except Exception as exc:
    print("Evaluation suite import could not run in this environment.")
    print(type(exc).__name__, exc)
    print("The suite is defined in src/evals/suite.py with 12 cases and three evaluator types.")

In [ ]:
# Requires API credentials and may take several minutes.
# from src.evals.suite import run_eval
# report = await run_eval()
# report

### Evaluation Results Snapshot

The repository includes an evaluation summary image from a prior run. Re-run the previous cell before final submission if code or prompts change.

In [ ]:
from IPython.display import Image, display

eval_image = Path('evals.png')
if eval_image.exists():
    display(Image(filename=str(eval_image)))
else:
    print("No evals.png found. Run the evaluation suite and save a summary before final submission.")

## Runnable Interfaces

The project can be run as either a web app or a CLI.

```bash
uv run shiny run app.py
```

```bash
uv run main.py
```

The Shiny app is the recommended demo interface because it shows the investigation stages and final story in a browser.

In [ ]:
from IPython.display import Image, display

ui_image = Path('front_end.png')
if ui_image.exists():
    display(Image(filename=str(ui_image)))
else:
    print("No front_end.png screenshot found.")

## Reflection

The system succeeds at splitting the noir story generation task into separate reasoning stages. Structured outputs keep the Detective and Narrator outputs predictable, and the Witness gives the generated story a grounding step through RAG. The orchestration also makes the workflow easier to test than a single prompt because each stage can be inspected independently.

The largest risk is retrieval quality. The Witness currently uses a simple relevance heuristic based on passage length, so a future version should inspect semantic scores from ChromaDB and use stronger thresholds. A second improvement would be to expand the behavioral evaluator so it records whether the Witness used corpus or web fallback and whether tool usage happened in the expected order.

For final submission, re-run the full engine and evaluation suite with valid API credentials, then confirm that the generated examples and `evals.png` reflect the latest code.

## AI Usage Disclosure

Generative AI tools were used to help draft, debug, and polish parts of this project. The team is responsible for the final code, explanations, and evaluation results. Prompts and tool usage should be documented in the README before submission according to the course academic integrity policy.